### Understanding TFT data and working with .json files for analysis


In [44]:
import json
import pandas as pd
import os

def load_json(path):
    if not os.path.exists(path):
        print(f"  [skipped] {path} not found")
        return []
    with open(path) as f:
        data = json.load(f)
    print(f"  Loaded {len(data):,} matches from {path}")
    return data

my_matches   = load_json("TFT-match-data-set-17--main/matches.json")
chall_matches = load_json("data/challenger_matches.json")

all_matches = my_matches + chall_matches
print(f"\nTotal: {len(all_matches):,} matches combined")

  Loaded 50 matches from TFT-match-data-set-17--main/matches.json
  Loaded 888 matches from data/challenger_matches.json

Total: 938 matches combined


### Extract stats into dataframe

In [45]:
def extract_stats(matches):
    rows = []
    for match in matches:
        if match["info"].get("tft_set_number") != 17:
            continue
        puuid = match["_my_puuid"]
        info = match["info"]
        me = next((p for p in info["participants"] if p["puuid"] == puuid), None)
        if me is None:
            continue
        rows.append({
            "match_id":    match["metadata"]["match_id"],
            "date":        pd.to_datetime(info["game_datetime"], unit="ms"),
            "player_name": match.get("_player_name", "me"),
            "is_me":       match.get("_is_me", True),
            "placement":   me["placement"],
            "level":       me["level"],
            "last_round":  me["last_round"],
            "gold_left":   me["gold_left"],
            "augments":    [a.replace("TFT_Augment_", "") for a in me.get("augments", [])],
            "traits":      me.get("traits", []),
            "units":       me.get("units", []),
        })
    return pd.DataFrame(rows)

df = extract_stats(all_matches)
print(f"{len(df):,} rows  |  {df['is_me'].sum()} my games  |  {(~df['is_me']).sum()} challenger games")
df.head()

932 rows  |  45 my games  |  887 challenger games


,match_id,date,player_name,is_me,placement,level,last_round,gold_left,augments,traits,units
0,NA1_5562880650,2026-05-18 04:53:11.410,me,True,4,9,34,0,[],"[{'name': 'TFT17_APTrait', 'num_units': 4, 'st...","[{'character_id': 'TFT17_Aatrox', 'itemNames':..."
1,NA1_5560691122,2026-05-15 05:37:09.012,me,True,6,9,31,7,[],"[{'name': 'TFT17_AnimaSquad', 'num_units': 1, ...","[{'character_id': 'TFT17_Nasus', 'itemNames': ..."
2,NA1_5559979852,2026-05-14 05:08:10.272,me,True,4,9,35,1,[],"[{'name': 'TFT17_APTrait', 'num_units': 4, 'st...","[{'character_id': 'TFT17_Aatrox', 'itemNames':..."
3,NA1_5559669327,2026-05-13 23:18:42.862,me,True,5,9,31,1,[],"[{'name': 'TFT17_AnimaSquad', 'num_units': 1, ...","[{'character_id': 'TFT17_IvernMinion', 'itemNa..."
4,NA1_5559167035,2026-05-13 03:01:23.733,me,True,5,8,31,19,[],"[{'name': 'TFT17_ADMIN', 'num_units': 2, 'styl...","[{'character_id': 'TFT17_Teemo', 'itemNames': ..."


### Encode unit rarity = costs (teemo = 1 cost, Karma = 4 cost, etc)

In [46]:
RARITY_TO_COST = {0: 1, 1: 2, 2: 3, 4: 4, 6: 5}

def parse_units(units_list):
    by_cost = {c: [] for c in range(1, 6)}
    itemized = []

    for u in units_list:
        # skip non-Set17 units (summons, old-set units)
        if not u["character_id"].startswith("TFT17_"):
            continue
        cost = RARITY_TO_COST.get(u["rarity"])
        if cost is None:
            continue
        name  = u["character_id"].split("_", 1)[-1]
        tier  = u.get("tier", 1)
        items = [i.replace("TFT_Item_", "").replace("TFT4_Item_", "") for i in u.get("itemNames", [])]

        unit_info = {"unit": name, "tier": tier, "items": items}
        by_cost[cost].append(unit_info)

        if items:
            itemized.append({**unit_info, "cost": cost, "n_items": len(items)})

    itemized.sort(key=lambda x: x["n_items"], reverse=True)
    return by_cost, itemized

parsed = df["units"].apply(parse_units)
df["_parsed_by_cost"] = parsed.apply(lambda x: x[0])
df["itemized_units"]  = parsed.apply(lambda x: x[1])

for cost in range(1, 6):
    df[f"units_{cost}cost"] = df["_parsed_by_cost"].apply(lambda d: [u["unit"] for u in d[cost]])
    df[f"n_{cost}cost"]     = df[f"units_{cost}cost"].apply(len)

df.drop(columns=["_parsed_by_cost"], inplace=True)

df[["match_id", "player_name", "placement", "itemized_units"]].head()

,match_id,player_name,placement,itemized_units
0,NA1_5562880650,me,4,"[{'unit': 'MissFortune', 'tier': 3, 'items': [..."
1,NA1_5560691122,me,6,"[{'unit': 'AurelionSol', 'tier': 2, 'items': [..."
2,NA1_5559979852,me,4,"[{'unit': 'MissFortune', 'tier': 3, 'items': [..."
3,NA1_5559669327,me,5,"[{'unit': 'Karma', 'tier': 2, 'items': ['Stati..."
4,NA1_5559167035,me,5,"[{'unit': 'Teemo', 'tier': 3, 'items': ['TFT17..."


In [ ]:
# Most commonly itemized units across all games
from collections import Counter

all_itemized = [u for game in df["itemized_units"] for u in game]
unit_counts = Counter(u["unit"] for u in all_itemized)

#pd.DataFrame(unit_counts.most_common(35), columns=["unit", "games_itemized"])

,unit,games_itemized
0,Nunu,265
1,Blitzcrank,249
2,TahmKench,228
3,Rammus,220
4,Riven,197
5,Corki,195
6,Jhin,183
7,Fiora,162
8,Karma,154
9,Vex,152


In [48]:
import requests as _requests

UNIT_DATA_PATH = "data/tft_units.json"

if not os.path.exists(UNIT_DATA_PATH):
    print("Fetching Set 17 unit data from CommunityDragon...")
    url = "https://raw.communitydragon.org/latest/cdragon/tft/en_us.json"
    raw = _requests.get(url, timeout=60).json()

    set17 = next(s for s in raw["setData"] if s["number"] == 17)
    units = [
        {
            "apiName": u["apiName"],
            "name":    u["name"],
            "cost":    u["cost"],
            "role":    u.get("role") or "",
            "traits":  u.get("traits", []),
        }
        for u in set17["champions"]
        if u["apiName"].startswith("TFT17_") and u.get("cost", 99) <= 5
    ]
    with open(UNIT_DATA_PATH, "w") as f:
        json.dump(units, f, indent=2)
    print(f"Saved {len(units)} units → {UNIT_DATA_PATH}")
else:
    with open(UNIT_DATA_PATH) as f:
        units = json.load(f)
    print(f"Loaded {len(units)} Set 17 units from cache")

UNIT_LOOKUP = {u["apiName"].replace("TFT17_", ""): u for u in units}

pd.DataFrame(units)[["name", "cost", "role", "traits"]]

Loaded 65 Set 17 units from cache


,name,cost,role,traits
0,Briar,1,ADFighter,"[Anima, Primordian, Rogue]"
1,Bel'Veth,2,ADFighter,"[Primordian, Challenger, Marauder]"
2,Miss Fortune,3,,"[Gun Goddess, Choose Trait]"
3,Akali,2,ADFighter,"[N.O.V.A., Marauder]"
4,Bard,5,APCaster,"[Meeple, Conduit]"
...,...,...,...,...
60,Rek'Sai,1,APTank,"[Primordian, Brawler]"
61,Pantheon,2,ADTank,"[Timebreaker, Brawler, Replicator]"
62,Morgana,4,APTank,[Dark Lady]
63,Mini Black Hole,1,APTank,[]


In [49]:
# Use CommunityDragon role strings directly
# Only override units with a missing/empty role field
ROLE_OVERRIDES = {
    "MissFortune": "ADCarry",  # role field is empty in CDragon
}

def get_unit_role(unit_name):
    if unit_name in ROLE_OVERRIDES:
        return ROLE_OVERRIDES[unit_name]
    unit = UNIT_LOOKUP.get(unit_name)
    if not unit:
        return "Other"
    return unit["role"] or "Other"

def label_roles(itemized_units):
    roles = {}
    for u in itemized_units:
        role = get_unit_role(u["unit"])
        if role not in roles:
            roles[role] = u["unit"]
    return roles

df["roles"] = df["itemized_units"].apply(label_roles)

# expand all CDragon roles into columns
all_roles = ["ADCarry", "ADCaster", "ADReaper", "ADSpecialist", "ADFighter", "ADTank",
             "APCarry", "APCaster", "APReaper", "APFighter", "APTank", "HFighter"]

for role in all_roles:
    df[role] = df["roles"].apply(lambda r: r.get(role))

df[["match_id", "placement"] + all_roles].head(15)

,match_id,placement,ADCarry,ADCaster,ADReaper,ADSpecialist,ADFighter,ADTank,APCarry,APCaster,APReaper,APFighter,APTank,HFighter
0,NA1_5562880650,4,MissFortune,NaN,NaN,NaN,NaN,NaN,NaN,Nami,NaN,Shen,Maokai,NaN
1,NA1_5560691122,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AurelionSol,NaN,Blitzcrank,Nunu,NaN
2,NA1_5559979852,4,MissFortune,NaN,NaN,NaN,NaN,NaN,NaN,Nami,NaN,NaN,Maokai,NaN
3,NA1_5559669327,5,NaN,NaN,NaN,NaN,NaN,NaN,Vex,Karma,NaN,Blitzcrank,Nunu,NaN
4,NA1_5559167035,5,NaN,NaN,NaN,NaN,NaN,NaN,Teemo,Lissandra,NaN,NaN,Leona,NaN
5,NA1_5558619798,3,NaN,Kaisa,NaN,NaN,Briar,Rhaast,NaN,NaN,NaN,NaN,RekSai,NaN
6,NA1_5558584521,2,NaN,NaN,NaN,NaN,NaN,NaN,Teemo,Lissandra,NaN,NaN,Leona,NaN
7,NA1_5557925908,4,NaN,NaN,Talon,Caitlyn,NaN,NaN,NaN,TwistedFate,NaN,NaN,Jax,NaN
8,NA1_5557523982,5,NaN,NaN,NaN,NaN,NaN,NaN,Teemo,Lissandra,NaN,NaN,Leona,NaN
9,NA1_5556979744,7,Jhin,Kaisa,NaN,NaN,NaN,Galio,NaN,Karma,NaN,NaN,Nunu,NaN


In [50]:
# XP needed to reach each level (cumulative).
# You earn 2 XP automatically per round; buying XP is 4 gold = 4 XP (1:1 ratio).
XP_THRESHOLDS = {2: 2, 3: 6, 4: 10, 5: 20, 6: 36, 7: 56, 8: 80, 9: 84}

def estimate_gold_on_xp(level, last_round):
    xp_needed  = XP_THRESHOLDS.get(level, 0)
    auto_xp    = last_round * 2
    xp_bought  = max(0, xp_needed - auto_xp)
    return xp_bought  # 1 XP = 1 gold

def carry_status(itemized_units):
    """
    Returns (status, carry_unit, carry_cost, carry_tier).
    1/2/3-cost carries: need 3-star to count as 'hit'
    4-cost carries:     2-star = 'hit', 3-star = 'exceptional'
    5-cost carries:     2-star = 'hit', 3-star = 'exceptional (extremely rare)'
    """
    if not itemized_units:
        return "no_carry", None, None, None

    carry = itemized_units[0]  # sorted by n_items desc — most itemized = main carry
    unit, cost, tier = carry["unit"], carry["cost"], carry["tier"]

    if cost <= 3:
        status = "hit" if tier >= 3 else "missed"
    else:  # 4 or 5 cost
        if tier >= 3:
            status = "exceptional"
        elif tier >= 2:
            status = "hit"
        else:
            status = "missed"

    return status, unit, cost, tier

df["gold_on_xp"] = df.apply(
    lambda r: estimate_gold_on_xp(r["level"], r["last_round"]), axis=1
)

carry_cols = df["itemized_units"].apply(
    lambda u: pd.Series(carry_status(u), index=["carry_status", "carry_unit", "carry_cost", "carry_tier"])
)
df = pd.concat([df, carry_cols], axis=1)

df["n_3stars"] = df["itemized_units"].apply(
    lambda units: sum(1 for u in units if u["tier"] >= 3)
)

df[["match_id", "placement", "last_round", "gold_left", "gold_on_xp",
    "carry_unit", "carry_cost", "carry_tier", "carry_status", "n_3stars"]].head(15)

,match_id,placement,last_round,gold_left,gold_on_xp,carry_unit,carry_cost,carry_tier,carry_status,n_3stars
0,NA1_5562880650,4,34,0,16,MissFortune,3.0,3.0,hit,2
1,NA1_5560691122,6,31,7,22,AurelionSol,4.0,2.0,hit,0
2,NA1_5559979852,4,35,1,14,MissFortune,3.0,3.0,hit,1
3,NA1_5559669327,5,31,1,22,Karma,4.0,2.0,hit,0
4,NA1_5559167035,5,31,19,18,Teemo,1.0,3.0,hit,4
5,NA1_5558619798,3,34,1,16,Briar,1.0,3.0,hit,3
6,NA1_5558584521,2,35,0,14,Teemo,1.0,3.0,hit,3
7,NA1_5557925908,4,33,0,14,Caitlyn,1.0,3.0,hit,4
8,NA1_5557523982,5,30,22,20,Teemo,1.0,3.0,hit,2
9,NA1_5556979744,7,27,9,30,Nunu,4.0,2.0,hit,0


### Sanity Checks Section:

In [51]:
df_csv = df.copy()
df_csv["augments"] = df_csv["augments"].apply(json.dumps)
df_csv["traits"] = df_csv["traits"].apply(json.dumps)
df_csv["units"] = df_csv["units"].apply(json.dumps)

df_csv.to_csv("data/tft_games.csv", index=False)
print(f"Saved {len(df_csv)} rows to data/tft_games.csv")

Saved 932 rows to data/tft_games.csv


In [52]:
df["player_name"].value_counts()


player_name
ACAD wasian        100
ACAD Dishsoap       99
FNC Filup           98
Msian Emilywang     95
CTG Marcel P        93
junglebook1         91
VIT setsuko         88
grea                84
CTG dankmemes01     82
VIT k3soju          57
me                  45
Name: count, dtype: int64